In [12]:
import os
import glob
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Cargar la API key de Groq desde .env
load_dotenv("../.env")
assert os.getenv("GROQ_API_KEY"), "No se encontró GROQ_API_KEY en .env"
print("API key cargada ✓")

API key cargada ✓


In [13]:
# Encuentra todos los archivos del corpus
ruta_corpus = Path("../rag/corpus")

archivos_md = list(ruta_corpus.rglob("*.md"))
archivos_pdf = list(ruta_corpus.rglob("*.pdf"))

print(f"Archivos .md encontrados: {len(archivos_md)}")
for f in archivos_md:
    print(f"  - {f.relative_to(ruta_corpus)}")

print(f"\nArchivos .pdf encontrados: {len(archivos_pdf)}")
for f in archivos_pdf:
    print(f"  - {f.relative_to(ruta_corpus)}")

Archivos .md encontrados: 12
  - 01_cinco_c_credito.md
  - 02_credit_scoring.md
  - 06_metricas_evaluacion.md
  - 10_data_leakage.md
  - 03_pd_lgd_ead_basilea.md
  - 08_random_forest_scoring.md
  - 04_variables_german_credit.md
  - 11_validacion_cruzada.md
  - 07_regresion_logistica_scoring.md
  - 12_sarc_colombia.md
  - 05_desbalance_clases.md
  - 09_encoding_categoricas.md

Archivos .pdf encontrados: 2
  - bcbs128_credit_risk.pdf
  - 20230817_capitulo_ii.pdf


In [14]:
documentos = []

# Cargar markdowns
for archivo in archivos_md:
    loader = TextLoader(str(archivo), encoding="utf-8")
    docs = loader.load()
    # Añadir metadata con la fuente
    for d in docs:
        d.metadata["fuente"] = archivo.name
        d.metadata["tipo"] = "corpus_generado"
    documentos.extend(docs)

# Cargar PDFs
for archivo in archivos_pdf:
    print(f"Cargando {archivo.name}... ", end="", flush=True)
    loader = PyPDFLoader(str(archivo))
    docs = loader.load()  # devuelve una lista, una entrada por página
    for d in docs:
        d.metadata["fuente"] = archivo.name
        d.metadata["tipo"] = "pdf_oficial"
    documentos.extend(docs)
    print(f"{len(docs)} páginas")

print(f"\nTotal de documentos cargados: {len(documentos)}")

Cargando bcbs128_credit_risk.pdf... 68 páginas
Cargando 20230817_capitulo_ii.pdf... 44 páginas

Total de documentos cargados: 124


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documentos)
print(f"Total de chunks: {len(chunks)}")
print(f"\nEjemplo de chunk:")
print(f"Fuente: {chunks[0].metadata['fuente']}")
print(f"Contenido (primeros 300 chars):\n{chunks[0].page_content[:300]}")

Total de chunks: 1499

Ejemplo de chunk:
Fuente: 01_cinco_c_credito.md
Contenido (primeros 300 chars):
# Las 5C del crédito

Las 5C del crédito son el marco tradicional que utilizan las entidades financieras para evaluar el riesgo crediticio de un solicitante. Son: carácter, capacidad, capital, colateral y condiciones.

**Carácter** se refiere a la historia y disposición del solicitante para cumplir 


In [5]:
print("Cargando modelo de embeddings (primera vez descarga ~120MB)...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Test rápido
vec_prueba = embeddings.embed_query("¿Qué es el credit scoring?")
print(f"Modelo cargado ✓")
print(f"Dimensión de los embeddings: {len(vec_prueba)}")

Cargando modelo de embeddings (primera vez descarga ~120MB)...


/tmp/ipykernel_221014/3971720775.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado ✓
Dimensión de los embeddings: 384


In [6]:
ruta_chroma = "../rag/chroma_db"

# Si ya existía de una corrida anterior, lo borramos para empezar limpio
import shutil
if os.path.exists(ruta_chroma):
    shutil.rmtree(ruta_chroma)
    print("Vector store anterior eliminado.")

print(f"Indexando {len(chunks)} chunks en ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=ruta_chroma,
    collection_name="credit_scoring"
)
print(f"Vector store creado ✓")
print(f"Total documentos indexados: {vectorstore._collection.count()}")

Indexando 1499 chunks en ChromaDB...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store creado ✓
Total documentos indexados: 1499


In [8]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8} 
)

def probar_retriever(pregunta):
    print(f"\n{'='*70}")
    print(f"Pregunta: {pregunta}")
    print(f"{'='*70}")
    
    docs = retriever.invoke(pregunta)
    for i, doc in enumerate(docs, 1):
        print(f"\n--- Chunk {i} (fuente: {doc.metadata.get('fuente', 'N/A')}) ---")
        print(doc.page_content[:400])
        if len(doc.page_content) > 400:
            print("...")

# Tres preguntas de prueba: una sobre el dataset, una sobre teoría, una sobre regulación
probar_retriever("¿Qué significa la variable estado_cuenta en el dataset?")
probar_retriever("¿Qué es la probabilidad de incumplimiento PD en Basilea?")
probar_retriever("¿Qué exige el SARC sobre los modelos de credit scoring?")


Pregunta: ¿Qué significa la variable estado_cuenta en el dataset?

--- Chunk 1 (fuente: bcbs128.pdf) ---
272 
Table 1 
Hedging Sets for Interest Rate Risk Positions Per Currency 
Remaining maturity 
or rate-adjustment 
frequency 
Sovereign-referenced 
interest rates 
Non-sovereign-
referenced interest rates 
One year or less X X 
Over one year to five 
years 
X X 
Over five years X X 
81. For underlying debt instruments (e.g. floati ng rate notes) or payment legs (e.g. 
floating rate legs of interest
...

--- Chunk 2 (fuente: bcbs128.pdf) ---
express each commodity position (spot plus forward) in terms of the standard unit of 
measurement (barrels, kilos, grams etc.). The net position in each commodity will then be 
converted at current spot rates into the national currency.  
718(L). Secondly, in order to capture forward gap and interest rate risk within a time-band 
(which, together, are sometimes referred to as curvature/spread risk
...

--- Chunk 3 (fuente: bcbs128.pdf) ---
in the

In [15]:
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Test rápido
test = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Responde solo con 'OK' si me lees."}],
    max_tokens=10,
)
print(f"Respuesta de prueba: {test.choices[0].message.content}")

Respuesta de prueba: OK


In [ ]:
PROMPT_TEMPLATE = """Eres un asistente experto en credit scoring y gestión de riesgo crediticio. Tu tarea es responder preguntas usando ÚNICAMENTE la información del contexto proporcionado.

Reglas estrictas:
1. Si la respuesta está en el contexto, respóndela de forma clara y concisa, en español.
2. Si la información no está en el contexto, di explícitamente: "No tengo información suficiente en el corpus para responder eso."
3. NO inventes datos, cifras ni referencias normativas que no estén en el contexto.
4. Cuando uses información del contexto, cita la fuente entre paréntesis al final de cada afirmación importante. Ejemplo: (fuente: 03_pd_lgd_ead_basilea.md)
5. Si el contexto tiene información contradictoria, menciónalo.

Contexto:
{contexto}

Pregunta: {pregunta}

Respuesta:"""


def responder(pregunta, k=8, verbose=False):
    # 1. Retrieval
    docs = vectorstore.similarity_search(pregunta, k=k)
    
    # 2. Armar el contexto con marcadores de fuente
    contexto = "\n\n---\n\n".join([
        f"[fuente: {d.metadata.get('fuente', 'desconocida')}]\n{d.page_content}"
        for d in docs
    ])
    
    if verbose:
        print(f"--- Chunks recuperados ({len(docs)}) ---")
        for i, d in enumerate(docs, 1):
            print(f"{i}. {d.metadata.get('fuente', 'N/A')}")
        print()
    
    # 3. Construir el prompt y llamar a Groq
    prompt = PROMPT_TEMPLATE.format(contexto=contexto, pregunta=pregunta)
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Eres un asistente experto que responde con precisión usando solo la información proporcionada."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,    # bajo = respuestas más deterministas y fieles al contexto
        max_tokens=600,
    )
    
    respuesta = response.choices[0].message.content
    fuentes = list({d.metadata.get("fuente", "N/A") for d in docs})
    
    return {
        "pregunta": pregunta,
        "respuesta": respuesta,
        "fuentes": fuentes,
        "chunks_usados": len(docs),
    }

In [18]:
preguntas_prueba = [
    # Pregunta del dominio del dataset
    "¿Qué significa la variable estado_cuenta en el dataset German Credit y por qué es predictiva?",
    
    # Pregunta sobre teoría/Basilea
    "¿Qué son PD, LGD y EAD según el enfoque IRB de Basilea II?",
    
    # Pregunta sobre regulación colombiana
    "¿Qué exige el SARC sobre los procesos de otorgamiento de crédito?",
    
    # Pregunta sobre metodología ML
    "¿Por qué la métrica accuracy no es adecuada para credit scoring con clases desbalanceadas?",
    
    # Pregunta que debería decir "no sé" (fuera del corpus)
    "¿Cuál es el precio actual del dólar en Colombia?",
]

for p in preguntas_prueba:
    resultado = responder(p, verbose=False)
    print(f"\n{'='*70}")
    print(f"P: {resultado['pregunta']}")
    print(f"{'='*70}")
    print(f"R: {resultado['respuesta']}")
    print(f"\nFuentes consultadas: {resultado['fuentes']}")


P: ¿Qué significa la variable estado_cuenta en el dataset German Credit y por qué es predictiva?
R: La variable `estado_cuenta` en el dataset German Credit se refiere al estado de la cuenta corriente del solicitante. Esta variable tiene cuatro categorías: A11 (saldo negativo), A12 (saldo entre 0 y 200 DM), A13 (saldo mayor a 200 DM) y A14 (sin cuenta corriente registrada). Es la variable más predictiva del dataset porque refleja la situación financiera actual del solicitante y su capacidad para gestionar sus finanzas, lo que puede influir en su capacidad para pagar un crédito (fuente: 04_variables_german_credit.md).

Fuentes consultadas: ['01_cinco_c_credito.md', '07_regresion_logistica_scoring.md', '08_random_forest_scoring.md', 'bcbs128.pdf', '04_variables_german_credit.md', '05_desbalance_clases.md']

P: ¿Qué son PD, LGD y EAD según el enfoque IRB de Basilea II?
R: Según el enfoque IRB de Basilea II, PD (Probability of Default) es la probabilidad de incumplimiento del deudor en un 